In [ ]:
!pip install arch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.3/981.3 kB 11.0 MB/s eta 0:00:00


In [ ]:
# Importing libraries
import pandas as pd
import plotly.express as px
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import statsmodels.api as sm
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.diagnostic import acorr_ljungbox
from arch import arch_model

# Loading and Filtering Data

In [ ]:
from google.colab import files
uploaded = files.upload()

import pandas as pd

crude_data = pd.read_csv("RWTCd.csv")
crude_data.head()

FileNotFoundError: [Errno 2] No such file or directory: 'RWTCd.csv'

In [ ]:
# Filtering the data
crude_data["Date"] = pd.to_datetime(crude_data["Date"])
crude_data = crude_data[crude_data["Date"] >= pd.Timestamp("2010-01-01")].reset_index(drop=True)
crude_data = crude_data[crude_data["Price"] > 0].copy()
crude_data= crude_data.dropna()
crude_data.head()

In [ ]:
# Calculating log returns
crude_data= crude_data.dropna()
crude_data["Log Returns"]= np.log(crude_data["Price"] / crude_data["Price"].shift(1))
crude_data= crude_data.dropna()
crude_data.head()

# Exploratory Data Analysis

In [ ]:
crude_data.size

In [ ]:
crude_data.isna().sum()

In [ ]:
crude_data.info()

In [ ]:
crude_data.describe()

In [ ]:
# Visualising the price over time
fig1= px.line(data_frame= crude_data, x= "Date", y= "Price", title= "Closing Price Data for Crude Oil")
fig1.show()

In [ ]:
# Visualising the distribution of daily log returns
plt.figure()
plt.hist(crude_data["Log Returns"].dropna(), bins=100)
plt.xlabel("Daily Log Returns")
plt.ylabel("Frequency")
plt.title("Distribution of Daily Log Returns")
plt.show()

In [ ]:
decompose = seasonal_decompose(crude_data["Log Returns"], period=12).plot()
#period: how many times calculated in a year (12=monthly, 4=quarterly, 1=annual)

In [ ]:
plt.figure(figsize=(5, 0.8))
crude_data["Log Returns"].rolling(window=12, center=True).mean().plot(lw=1)
plt.margins(x=0)
plt.ylabel('Returns')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

plt.figure(figsize=(5, 1.35))

# 1. Calculate rolling mean and ensure it is a Series
rolling_series = crude_data["Log Returns"].rolling(window=30, center=True).mean().squeeze()

# 2. Plot the main data (lw=1)
rolling_series.plot(lw=1, label='Rolling Mean')

# --- ADDING THE RED TRENDLINE ---
y_values = rolling_series.dropna()
# polyfit needs numerical x-values (0, 1, 2...)
x_numeric = np.arange(len(y_values))

z = np.polyfit(x_numeric, y_values.to_numpy().flatten(), 1)
p = np.poly1d(z)

# Plot trendline using the datetime index of the non-NaN values
plt.plot(y_values.index, p(x_numeric), color="red", lw=1)

# --- X-AXIS YEAR FORMATTING ---
ax = plt.gca()
# Sets the locator to find the start of each year
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_locator(ticker.MaxNLocator(8))
# Formats the tick label to show only the year (e.g., 2021)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.margins(x=0)
plt.ylabel('Returns')
plt.xticks(rotation=0) # Keeps years horizontal for readability
plt.tight_layout()
plt.show()

# Assumption Testing

In [ ]:
# ADF Test

def adf_test(timeseries):
    print("Results of Dickey-Fuller Test:")
    dftest = adfuller(timeseries, autolag="AIC")
    dfoutput = pd.Series(
        dftest[0:4],
        index=[
            "Test Statistic",
            "p-value",
            "#Lags Used",
            "Number of Observations Used",
        ],
    )
    for key, value in dftest[4].items():
        dfoutput["Critical Value (%s)" % key] = value
    print(dfoutput)

In [ ]:
series = crude_data["Log Returns"]
output = adf_test(series)

In [ ]:
# Ljung-Box Test

squared_returns = series**2 # using squared_return to check variance dependence i.e. if a big shock today leads to a big shock tomorrow
lb_results = acorr_ljungbox(squared_returns, lags=[10, 15, 20, 100], return_df=True) # testing up to 10 lags to see short-to-medium term persistence
lb_results


# GJR-GARCH Implementation

In [ ]:
scaled_returns = 100 * crude_data['Log Returns']

In [ ]:
# 1. Define the combinations (p, o, q)
# The order is (p=GARCH lag, o=GJR/Asymmetry lag, q=ARCH lag)
combinations = [
    (1, 1, 1), (2, 1, 1), (1, 2, 1), (1, 1, 2),
    (2, 2, 1), (2, 1, 2), (1, 2, 2), (2, 2, 2)
]

# 2. Store results in a list
results_list = []

# 3. Loop through combinations
for p, o, q in combinations:
    try:
        # Fit the model
        model = arch_model(scaled_returns, vol='GARCH', p=p, o=o, q=q, dist='t')
        res = model.fit(disp='off')

        # Save the scores
        results_list.append({
            'Model': f'({p},{o},{q})',
            'AIC': res.aic,
            'BIC': res.bic,
        })
    except Exception as e:
        print(f"Model ({p},{o},{q}) failed to converge: {e}")

# 4. Display as a DataFrame
comparison_df = pd.DataFrame(results_list)
comparison_df = comparison_df.sort_values(by='AIC') # Sort by best AIC
print(comparison_df)

In [ ]:
# Fit the GJR-GARCH(1,1,1) model
model_gjr_111 = arch_model(scaled_returns, vol='GARCH', p=1, o=1, q=1, dist='t')
results_gjr_111 = model_gjr_111.fit(disp='off')

# Print the summary
print(results_gjr_111.summary())

# Validating the Model using The Ljung Box Test

In [ ]:
std_resid_sq = results_gjr_111.std_resid ** 2
lb_test = acorr_ljungbox(std_resid_sq, lags=[5, 10, 15, 20,100], return_df=True)
print("Ljung-Box Test for Squared Residuals:")
print(lb_test)

# Generating a Conditional Volatility Forecast

In [ ]:
# 1. Extract parameters for the Long-Run Volatility calculation
omega = results_gjr_111.params['omega']
alpha = results_gjr_111.params['alpha[1]']
gamma = results_gjr_111.params['gamma[1]']
beta = results_gjr_111.params['beta[1]']

# Calculation for GJR-GARCH Unconditional Volatility
# (Note: 0.5 * gamma accounts for the probability of negative shocks)
unconditional_var = omega / (1 - alpha - beta - 0.5 * gamma)
unconditional_vol = np.sqrt(unconditional_var)

# 2. Generate the 10-day Forecast
horizon = 10
forecasts = results_gjr_111.forecast(horizon=horizon, reindex=False)
pred_vol = np.sqrt(forecasts.variance.iloc[-1])

# 3. Create the Forecast Table
forecast_steps = [f'Day {i+1}' for i in range(horizon)]
forecast_table = pd.DataFrame({
    'Step': forecast_steps,
    'Projected Volatility (%)': pred_vol.values.round(4),
    'Vs. Long-Run Avg (%)': (pred_vol.values - unconditional_vol).round(4)
})

# 4. Visualization
plt.figure(figsize=(12, 6))

# Plot the 10-day trend
plt.plot(forecast_steps, pred_vol, marker='o', color='darkorange',
         linewidth=2.5, label='Conditional Volatility Forecast')

# Add the Long-Run Volatility Line
plt.axhline(y=unconditional_vol, color='navy', linestyle='--',
            linewidth=2, label=f'Long-Run Average ({unconditional_vol:.2f}%)')

# Formatting the Chart
plt.title("WTI Crude Oil: 10-Day Volatility Mean Reversion Path", fontsize=14, fontweight='bold')
plt.ylabel("Volatility (Standard Deviation %)")
plt.xlabel("Forecast Horizon")
plt.grid(axis='y', linestyle=':', alpha=0.7)
plt.legend()

# Display results
print("--- WTI VOLATILITY FORECAST TABLE ---")
print(forecast_table.to_string(index=False))
print(f"\nModel Unconditional Volatility: {unconditional_vol:.4f}%")
plt.show()

In [ ]:
from scipy.stats import t
import numpy as np

alpha = 0.05
horizon = 10

# Use fitted model
model_wti = results_gjr_111
params = model_wti.params

# ---- Parameters ----
mu = params["mu"]

# Robust DOF extraction
if "nu" in params:
    nu = params["nu"]
elif "eta" in params:
    nu = params["eta"]
else:
    raise ValueError("Degrees of freedom parameter not found")

# ---- Forecast ----
fc = model_wti.forecast(horizon=horizon, reindex=False)

# Mean
mu_10 = float(np.sum(fc.mean.values[-1]))

# Variance aggregation
horizon_variances = fc.variance.values[-1]
sigma_10 = float(np.sqrt(np.sum(horizon_variances)))

# ---- Standardized Student-t scaling ----
scale = np.sqrt((nu - 2) / nu)

t_alpha = t.ppf(alpha, df=nu)
z_alpha = scale * t_alpha

# ---- VaR (loss form, positive) ----
VaR_10_loss = -(mu_10 + sigma_10 * z_alpha)

# ---- ES ----
pdf_alpha = t.pdf(t_alpha, df=nu)

ES_10_loss = sigma_10 * (
    scale * ((nu + t_alpha**2) / ((nu - 1) * alpha)) * pdf_alpha
) - mu_10

print("\n--- WTI CRUDE OIL ---")
print(f"10-day 95% VaR (loss): {VaR_10_loss:.4f}%")
print(f"10-day 95% ES  (loss): {ES_10_loss:.4f}%")

WTI crude oil returns were modeled using a GJR-GARCH specification with a Student’s t distribution to capture the pronounced volatility clustering and heavy tails characteristic of commodity markets. The 10-day ahead 95% VaR is estimated at 11.91%, indicating a relatively high level of downside risk over the forecast horizon. The corresponding Expected Shortfall is 16.67%, suggesting that when losses exceed the VaR threshold, they tend to be considerably larger on average. The substantial difference between VaR and ES reflects significant tail risk, consistent with the tendency of oil markets to experience large price swings due to geopolitical events, supply disruptions, and macroeconomic uncertainty. The Student-t distribution effectively captures these extreme movements by allowing for a higher likelihood of large deviations compared to a normal framework.